Notebook 3 - Capital Structure & LBO Engine

This notebook models:
- ARR-based valuation
- Growth sensitivity
- Capital allocation scenarios
- Debt / equity structure
- 5-year exit modeling
- LBO simulation with IRR & MOIC

Designed for Strategic Finance & Corporate Development use.

In [ ]:
import numpy as np
import pandas as pd


In [ ]:
# 1. BASE VALUATION ENGINE (ARR MULTIPLE)
starting_arr = 49_323_347.24
growth_rate = 0.40


def implied_multiple(growth):
    if growth < 0.2:
        return 4
    elif growth < 0.4:
        return 6
    elif growth < 0.6:
        return 8
    else:
        return 10


base_multiple = implied_multiple(growth_rate)
base_enterprise_value = starting_arr * base_multiple

print("Base Enterprise Value:", base_enterprise_value)

Base Enterprise Value: 394586777.92


In [ ]:
# 2. GROWTH SENSITIVITY ANALYSIS
growth_scenarios = np.linspace(0.2, 0.7, 6)

valuation_results = []

for g in growth_scenarios:
    multiple = implied_multiple(g)
    value = starting_arr * multiple
    valuation_results.append((g, multiple, value))

valuation_table = pd.DataFrame(
    valuation_results,
    columns=["Growth Rate", "Multiple", "Enterprise Value"]
)

print("\nGrowth Sensitivity Table")
print(valuation_table)



Growth Sensitivity Table
   Growth Rate  Multiple  Enterprise Value
0          0.2         6      2.959401e+08
1          0.3         6      2.959401e+08
2          0.4         8      3.945868e+08
3          0.5         8      3.945868e+08
4          0.6        10      4.932335e+08
5          0.7        10      4.932335e+08


In [ ]:
# 3. CAPITAL ALLOCATION SCENARIOS
multiple = 8  # market multiple assumption

arr_acquisition = starting_arr + (5 * 2.5 * 1_000_000)
arr_retention = starting_arr * 1.08
arr_expansion = starting_arr * 1.12

capital_results = {
    "Acquisition": arr_acquisition * multiple,
    "Retention": arr_retention * multiple,
    "Expansion": arr_expansion * multiple
}

print("\nCapital Allocation Impact")
print(capital_results)


Capital Allocation Impact
{'Acquisition': 494586777.92, 'Retention': 426153720.15360004, 'Expansion': 441937191.27040005}


In [ ]:
# 4. CAPITAL STRUCTURE
debt_ratio = 0.60
equity_ratio = 0.40

debt = base_enterprise_value * debt_ratio
equity = base_enterprise_value * equity_ratio

print("\nCapital Structure")
print("Debt:", debt)
print("Equity:", equity)


Capital Structure
Debt: 236752066.752
Equity: 157834711.168


In [ ]:
# 5. EXIT SCENARIO (ARR-BASED)
exit_growth = 0.35
years = 5
exit_multiple = 7

arr_projection = starting_arr

for _ in range(years):
    arr_projection *= (1 + exit_growth)

exit_enterprise_value_arr = arr_projection * exit_multiple
exit_equity_value_arr = exit_enterprise_value_arr - debt

moic_arr = exit_equity_value_arr / equity
irr_arr = moic_arr ** (1 / years) - 1

print("\nARR-Based Exit")
print("MOIC:", moic_arr)
print("IRR:", irr_arr)


ARR-Based Exit
MOIC: 8.308823144531251
IRR: 0.5272421551202264


In [ ]:
# 6. LBO SIMULATION (EBITDA-BASED EXIT)
ebitda_margin = 0.30
tax_rate = 0.25
interest_rate = 0.08
amortization_rate = 0.10

revenue = starting_arr
debt_balance = debt

revenues = []
ebitdas = []
debts = []
fcfs = []

for year in range(years):

    revenue *= (1 + exit_growth)
    ebitda = revenue * ebitda_margin

    interest = debt_balance * interest_rate
    amortization = debt_balance * amortization_rate

    taxable_income = ebitda - interest
    taxes = max(taxable_income * tax_rate, 0)

    fcf = ebitda - interest - taxes - amortization
    debt_balance -= amortization

    revenues.append(revenue)
    ebitdas.append(ebitda)
    debts.append(debt_balance)
    fcfs.append(fcf)

exit_ebitda = ebitdas[-1]
exit_enterprise_value_ebitda = exit_ebitda * exit_multiple
exit_equity_value_ebitda = exit_enterprise_value_ebitda - debts[-1]

moic_lbo = exit_equity_value_ebitda / equity
irr_lbo = moic_lbo ** (1 / years) - 1

print("\nLBO Exit (EBITDA-Based)")
print("MOIC:", moic_lbo)
print("IRR:", irr_lbo)


LBO Exit (EBITDA-Based)
MOIC: 2.0569119433593754
IRR: 0.15516265430187315
